In [4]:

# 0. Import packages

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [5]:

# 1. Read official DfT 2024 road safety datasets


collision_url = "https://data.dft.gov.uk/road-accidents-safety-data/dft-road-casualty-statistics-collision-2024.csv"
vehicle_url = "https://data.dft.gov.uk/road-accidents-safety-data/dft-road-casualty-statistics-vehicle-2024.csv"
casualty_url = "https://data.dft.gov.uk/road-accidents-safety-data/dft-road-casualty-statistics-casualty-2024.csv"

collisions = pd.read_csv(collision_url, low_memory=False)
vehicles = pd.read_csv(vehicle_url, low_memory=False)
casualties = pd.read_csv(casualty_url, low_memory=False)

print("Collisions:", collisions.shape)
print("Vehicles:", vehicles.shape)
print("Casualties:", casualties.shape)

Collisions: (100927, 44)
Vehicles: (183514, 32)
Casualties: (128272, 23)


Because we want to build the modle,I load the three official 2024 road safety datasets: collisions, vehicles, and casualties. These datasets are stored separately because they describe different levels of information. The collisions table records one row per collision, while the vehicles and casualties tables record one row per vehicle and casualty respectively.

The displayed table shapes show how many records and variables are included in each dataset. This is useful because my research question is focused on collision severity, so I need to understand how these three datasets can be combined into one modelling table.



In [6]:

# 2. Basic key checks


for name, df in {
    "collisions": collisions,
    "vehicles": vehicles,
    "casualties": casualties
}.items():
    print("\n", name)
    print("collision_index:", "collision_index" in df.columns)
    print("collision_year:", "collision_year" in df.columns)
    print("collision_ref_no:", "collision_ref_no" in df.columns)


 collisions
collision_index: True
collision_year: True
collision_ref_no: True

 vehicles
collision_index: True
collision_year: True
collision_ref_no: True

 casualties
collision_index: True
collision_year: True
collision_ref_no: True


Before merging the datasets, I check whether the key variable `collision_index` exists in all three tables(similar to ‘LSOA’ as the merged key in spatial analysis). The output shows that it is available in the collisions, vehicles, and casualties datasets. Therefore, I can use this variable as the merge key when constructing the final collision-level dataset.

In [7]:

# 3. Clean collisions table


df_col = collisions.copy()

# Keep valid collision severity only
# 1 = Fatal, 2 = Serious, 3 = Slight
df_col = df_col[df_col["collision_severity"].isin([1, 2, 3])].copy()

# Binary target:
# severe = 1 means fatal or serious；severe = 0 means slight
df_col["severe"] = df_col["collision_severity"].isin([1, 2]).astype(int)

print("After severity filtering:", df_col.shape)
print(df_col["severe"].value_counts())
print(df_col["severe"].value_counts(normalize=True))

After severity filtering: (100927, 45)
severe
0    75858
1    25069
Name: count, dtype: int64
severe
0    0.751613
1    0.248387
Name: proportion, dtype: float64


In this step, I create the target variable for the machine learning task. The original variable `collision_severity` has three categories: fatal, serious, and slight. Since this project focuses on identifying severe collisions, I recode fatal and serious collisions as `severe = 1`, and slight collisions as `severe = 0`.

The output shows that severe collisions are less common than slight collisions. Therefore, later model evaluation  'should not rely only on accuracy' is resonable.

In [8]:

#  Create time-related variables


df_col["date"] = pd.to_datetime(df_col["date"], errors="coerce", dayfirst=True)

df_col["hour"] = pd.to_datetime(
    df_col["time"],
    format="%H:%M",
    errors="coerce"
).dt.hour

df_col["month"] = df_col["date"].dt.month

# STATS19 day_of_week: usually 1 = Sunday, 2 = Monday, ..., 7 = Saturday
df_col["weekend"] = df_col["day_of_week"].isin([1, 7]).astype(int)

# Night time: late night / early morning
df_col["night"] = df_col["hour"].isin([0, 1, 2, 3, 4, 5, 22, 23]).astype(int)

# Common commuting peak hours
df_col["peak_hour"] = df_col["hour"].isin([7, 8, 9, 16, 17, 18]).astype(int)

df_col[["date", "time", "hour", "month", "day_of_week", "weekend", "night", "peak_hour"]].head()

,date,time,hour,month,day_of_week,weekend,night,peak_hour
0,2024-12-05,16:10,16,12,5,0,0,1
1,2024-10-22,14:56,14,10,3,0,0,0
2,2024-01-24,17:50,17,1,4,0,0,1
3,2024-05-22,17:45,17,5,4,0,0,1
4,2024-08-11,15:36,15,8,1,1,0,0


In this step, I clean the date and time variables and create additional temporal features, including `hour`, `month`, `weekend`, `night`, and `peak_hour`. These variables are useful because collision severity may vary by time of day, day of week, or travel pattern.



In [9]:

# Check what would happen if all unknown / missing codes were dropped based on guidance book


drop_simulation_map = {
    "speed_limit": [-1],
    "urban_or_rural_area": [3],
    "weather_conditions": [9],
    "road_surface_conditions": [-1, 9],
    "light_conditions": [-1, 7],
    "road_type": [9],
    "junction_detail": [-1, 99],
    "junction_control": [-1, 9],
    "pedestrian_crossing": [-1, 99],
    "special_conditions_at_site": [-1, 9],
    "carriageway_hazards": [-1, 99],
    "trunk_road_flag": [-1]
}

df_drop_sim = df_col.copy()
drop_simulation_report = []

for col, values in drop_simulation_map.items():
    if col in df_drop_sim.columns:
        before = len(df_drop_sim)
        affected = df_drop_sim[col].isin(values).sum()

        # simulate dropping those rows
        df_drop_sim = df_drop_sim[~df_drop_sim[col].isin(values)].copy()
        after = len(df_drop_sim)

        drop_simulation_report.append({
            "column": col,
            "codes_that_would_be_dropped": values,
            "rows_matching_codes_before_drop": int(affected),
            "rows_removed_at_this_step": before - after,
            "remaining_rows_after_drop": after
        })

drop_simulation_report = pd.DataFrame(drop_simulation_report)

print("Original rows:", len(df_col))
print("Rows remaining if all listed unknown/missing codes were dropped:", len(df_drop_sim))
print("Total rows that would be removed:", len(df_col) - len(df_drop_sim))
print("Percentage removed:", round((len(df_col) - len(df_drop_sim)) / len(df_col) * 100, 2), "%")

drop_simulation_report

Original rows: 100927
Rows remaining if all listed unknown/missing codes were dropped: 22286
Total rows that would be removed: 78641
Percentage removed: 77.92 %


,column,codes_that_would_be_dropped,rows_matching_codes_before_drop,rows_removed_at_this_step,remaining_rows_after_drop
0,speed_limit,[-1],3,3,100924
1,urban_or_rural_area,[3],3,3,100921
2,weather_conditions,[9],3145,3145,97776
3,road_surface_conditions,"[-1, 9]",705,705,97071
4,light_conditions,"[-1, 7]",1253,1253,95818
5,road_type,[9],1593,1593,94225
6,junction_detail,"[-1, 99]",5656,5656,88569
7,junction_control,"[-1, 9]",40422,40422,48147
8,pedestrian_crossing,"[-1, 99]",662,662,47485
9,special_conditions_at_site,"[-1, 9]",24231,24231,23254


Before removing or recoding any categorical values, I inspect the frequency distribution of key coded variables, because many road safety variables use numeric codes, and values such as `0`, `-1`, `9`, or `99` do not always mean the same thing across variables.

This prevents  from incorrectly deleting valid categories such as `0 = None` or `0 = Not at junction`.

Before cleaning the coded categorical variables, I first check how many rows would be affected if missing or unknown codes were removed. The table shows that quite large amount (77.92 %) data is affected and some variables contain a large number of missing or unknown values, especially `special_conditions_at_site` and `junction_control`. Removing all of these rows would substantially reduce the dataset and could introduce selection bias.

Therefore, I do not delete all missing or unknown categorical codes. Instead, I remove only clearly invalid values with very small counts, such as missing speed limit and unallocated urban/rural area. For categorical variables with many unknown or missing codes, I recode them into a separate `-99` unknown category.


*This simulation is applied only to the collisions dataset because the collisions table is the main table and defines the modelling unit: one row represents one collision. The vehicles and casualties datasets are not included in this simulation because they are recorded at different levels, with one row per vehicle and one row per casualty（these two datasets are aggregated to collision level later before merging）.

In [10]:

# Based on the official DfT data guide:
# 0 is often a valid category, e.g. None / Not at junction / No crossing facility.
# -1 = Data missing or out of range
# 9 or 99 = unknown / self reported


df_col_clean = df_col.copy()

cleaning_report = []

# Drop invalid speed_limit = -1 only
before = len(df_col_clean)
df_col_clean = df_col_clean[df_col_clean["speed_limit"] != -1].copy()
after = len(df_col_clean)

cleaning_report.append({
    "column": "speed_limit",
    "action": "drop invalid rows",
    "values": [-1],
    "rows_affected": before - after,
    "remaining_rows": after
})

# Keep only urban / rural area 1 and 2
# 1 = Urban, 2 = Rural, 3 = Unallocated / unknown
before = len(df_col_clean)
df_col_clean = df_col_clean[df_col_clean["urban_or_rural_area"].isin([1, 2])].copy()
after = len(df_col_clean)

cleaning_report.append({
    "column": "urban_or_rural_area",
    "action": "drop invalid rows",
    "values": ["not in [1, 2]"],
    "rows_affected": before - after,
    "remaining_rows": after
})

# Recode unknown / missing categorical values as -99
unknown_map = {
    "weather_conditions": [9],
    "road_surface_conditions": [-1, 9],
    "light_conditions": [-1, 7],
    "road_type": [9],
    "junction_detail": [-1, 99],
    "junction_control": [-1, 9],
    "pedestrian_crossing": [-1, 99],
    "special_conditions_at_site": [-1, 9],
    "carriageway_hazards": [-1, 99],
    "trunk_road_flag": [-1]
}

for col, values in unknown_map.items():
    if col in df_col_clean.columns:
        n_affected = int(df_col_clean[col].isin(values).sum())
        df_col_clean[col] = df_col_clean[col].replace(values, -99)

        cleaning_report.append({
            "column": col,
            "action": "recode as unknown_or_missing category",
            "values": values,
            "rows_affected": n_affected,
            "remaining_rows": len(df_col_clean)
        })

cleaning_report = pd.DataFrame(cleaning_report)

print("Original collision rows:", len(df_col))
print("Cleaned collision rows:", len(df_col_clean))
cleaning_report

Original collision rows: 100927
Cleaned collision rows: 100921


,column,action,values,rows_affected,remaining_rows
0,speed_limit,drop invalid rows,[-1],3,100924
1,urban_or_rural_area,drop invalid rows,"[not in [1, 2]]",3,100921
2,weather_conditions,recode as unknown_or_missing category,[9],3145,100921
3,road_surface_conditions,recode as unknown_or_missing category,"[-1, 9]",2374,100921
4,light_conditions,recode as unknown_or_missing category,"[-1, 7]",1685,100921
5,road_type,recode as unknown_or_missing category,[9],2567,100921
6,junction_detail,recode as unknown_or_missing category,"[-1, 99]",7007,100921
7,junction_control,recode as unknown_or_missing category,"[-1, 9]",44485,100921
8,pedestrian_crossing,recode as unknown_or_missing category,"[-1, 99]",4046,100921
9,special_conditions_at_site,recode as unknown_or_missing category,"[-1, 9]",61967,100921


The cleaning report shows that only a small number(6rows) of rows were removed, while larger numbers of unknown or missing categorical values were recoded. This confirms that the dataset has been cleaned without unnecessarily losing a large proportion of collision records.

In [11]:

#  Create simplified derived collision-level indicators

# At junction:
# junction_control = 0 means "Not at junction or within 20 metres"
# -99 means unknown/missing
if "junction_control" in df_col_clean.columns:
    df_col_clean["at_junction"] = (df_col_clean["junction_control"] != 0).astype(int)
    df_col_clean.loc[df_col_clean["junction_control"] == -99, "at_junction"] = -99

# Pedestrian crossing facility:
# pedestrian_crossing = 0 means no physical crossing facility within 50m
# -99 means unknown/missing，i create this due to
if "pedestrian_crossing" in df_col_clean.columns:
    df_col_clean["has_pedestrian_crossing_facility"] = (
        ~df_col_clean["pedestrian_crossing"].isin([0, -99])
    ).astype(int)
    df_col_clean.loc[
        df_col_clean["pedestrian_crossing"] == -99,
        "has_pedestrian_crossing_facility"
    ] = -99

# Special condition:
# special_conditions_at_site = 0 means none
# -99 means unknown/missing
if "special_conditions_at_site" in df_col_clean.columns:
    df_col_clean["has_special_condition"] = (
        ~df_col_clean["special_conditions_at_site"].isin([0, -99])
    ).astype(int)
    df_col_clean.loc[
        df_col_clean["special_conditions_at_site"] == -99,
        "has_special_condition"
    ] = -99

# Carriageway hazard:
# carriageway_hazards = 0 means none
# -99 means unknown/missing
if "carriageway_hazards" in df_col_clean.columns:
    df_col_clean["has_carriageway_hazard"] = (
        ~df_col_clean["carriageway_hazards"].isin([0, -99])
    ).astype(int)
    df_col_clean.loc[
        df_col_clean["carriageway_hazards"] == -99,
        "has_carriageway_hazard"
    ] = -99

df_col_clean.head()

,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,local_authority_district,local_authority_ons_district,local_authority_highway,local_authority_highway_current,first_road_class,first_road_number,road_type,speed_limit,junction_detail_historic,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control_historic,pedestrian_crossing_physical_facilities_historic,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,severe,hour,month,weekend,night,peak_hour,at_junction,has_pedestrian_crossing_facility,has_special_condition,has_carriageway_hazard
0,202417M119024,2024,17M119024,450057,519938,-1.22722,54.57219,17,3,2,1,2024-12-05,5,16:10,-1,E06000002,E06000002,E06000002,6,0,6,30,-1,0,-99,-1,-1,-1,-1,0,4,2,2,-99,-1,0,1,3,2,E01035190,3,1,0.000000,1.000000,0,16,12,0,0,1,-99,0,-99,0
1,202417S312124,2024,17S312124,445858,516830,-1.29265,54.54466,17,3,1,1,2024-10-22,3,14:56,-1,E06000004,E06000004,E06000004,6,0,6,30,-1,0,-99,-1,-1,-1,-1,0,1,1,1,-99,-1,0,1,3,2,E01012239,3,1,0.000000,1.000000,0,14,10,0,0,0,-99,0,-99,0
2,2024070110901,2024,070110901,364765,390979,-2.53157,53.41443,7,3,2,1,2024-01-24,4,17:50,-1,E06000007,E06000007,E06000007,6,0,6,30,3,13,4,6,0,0,0,0,4,1,1,0,0,0,2,2,2,E01012459,-1,0,0.014743,0.985257,0,17,1,0,0,1,1,0,0,0
3,2024041446676,2024,041446676,365989,419804,-2.51630,53.67359,4,2,1,1,2024-05-22,4,17:45,-1,E06000008,E06000008,E06000008,3,675,6,50,-1,0,-99,-1,-1,-1,-1,0,1,5,5,-99,-1,0,2,1,2,E01012628,6,1,1.000000,0.000000,1,17,5,0,0,1,-99,0,-99,0
4,2024041478641,2024,041478641,366864,430071,-2.50416,53.76592,4,2,2,1,2024-08-11,1,15:36,-1,E06000008,E06000008,E06000008,3,6119,3,50,-1,16,4,6,0,-1,-1,0,1,1,1,-99,-1,0,1,1,2,E01012581,7,1,1.000000,0.000000,1,15,8,1,0,0,1,0,-99,0


In [12]:

#  Aggregate vehicles table to collision level


veh = vehicles.copy()

# Clean numeric age variables
if "age_of_driver" in veh.columns:
    veh["age_of_driver_clean"] = pd.to_numeric(veh["age_of_driver"], errors="coerce")
    veh.loc[veh["age_of_driver_clean"] < 0, "age_of_driver_clean"] = np.nan

if "age_of_vehicle" in veh.columns:
    veh["age_of_vehicle_clean"] = pd.to_numeric(veh["age_of_vehicle"], errors="coerce")
    veh.loc[veh["age_of_vehicle_clean"] < 0, "age_of_vehicle_clean"] = np.nan

veh_group = veh.groupby("collision_index")

veh_features = pd.DataFrame({
    "collision_index": veh_group.size().index,
    "vehicle_rows": veh_group.size().values
})

# Number of unique vehicle types involved
if "vehicle_type" in veh.columns:
    veh_features = veh_features.merge(
        veh_group["vehicle_type"].nunique().reset_index(name="n_vehicle_types"),
        on="collision_index",
        how="left"
    )

    # Approximate vulnerable vehicle involvement:
    # 1 = pedal cycle; 2-5 = motorcycle categories in common STATS19 coding
    veh["has_vulnerable_vehicle"] = veh["vehicle_type"].isin([1, 2, 3, 4, 5]).astype(int)

    veh_features = veh_features.merge(
        veh_group["has_vulnerable_vehicle"].max().reset_index(),
        on="collision_index",
        how="left"
    )

# Driver age features
if "age_of_driver_clean" in veh.columns:
    veh_features = veh_features.merge(
        veh_group["age_of_driver_clean"].mean().reset_index(name="mean_driver_age"),
        on="collision_index",
        how="left"
    )

    veh_features = veh_features.merge(
        veh_group["age_of_driver_clean"].min().reset_index(name="min_driver_age"),
        on="collision_index",
        how="left"
    )

# Vehicle age
if "age_of_vehicle_clean" in veh.columns:
    veh_features = veh_features.merge(
        veh_group["age_of_vehicle_clean"].mean().reset_index(name="mean_vehicle_age"),
        on="collision_index",
        how="left"
    )

# Vehicle movement / impact features
if "skidding_and_overturning" in veh.columns:
    # 0 = None
    # -1 = missing/out of range
    veh["has_skidding_or_overturning"] = (
        ~veh["skidding_and_overturning"].isin([0, -1])
    ).astype(int)

    veh_features = veh_features.merge(
        veh_group["has_skidding_or_overturning"].max().reset_index(),
        on="collision_index",
        how="left"
    )

if "vehicle_leaving_carriageway" in veh.columns:
    veh["has_vehicle_leaving_carriageway"] = (
        ~veh["vehicle_leaving_carriageway"].isin([0, -1])
    ).astype(int)

    veh_features = veh_features.merge(
        veh_group["has_vehicle_leaving_carriageway"].max().reset_index(),
        on="collision_index",
        how="left"
    )

if "hit_object_in_carriageway" in veh.columns:
    veh["has_hit_object_in_carriageway"] = (
        ~veh["hit_object_in_carriageway"].isin([0, -1])
    ).astype(int)

    veh_features = veh_features.merge(
        veh_group["has_hit_object_in_carriageway"].max().reset_index(),
        on="collision_index",
        how="left"
    )

if "hit_object_off_carriageway" in veh.columns:
    veh["has_hit_object_off_carriageway"] = (
        ~veh["hit_object_off_carriageway"].isin([0, -1])
    ).astype(int)

    veh_features = veh_features.merge(
        veh_group["has_hit_object_off_carriageway"].max().reset_index(),
        on="collision_index",
        how="left"
    )

print("Vehicle features:", veh_features.shape)
veh_features.head()

Vehicle features: (100927, 11)


,collision_index,vehicle_rows,n_vehicle_types,has_vulnerable_vehicle,mean_driver_age,min_driver_age,mean_vehicle_age,has_skidding_or_overturning,has_vehicle_leaving_carriageway,has_hit_object_in_carriageway,has_hit_object_off_carriageway
0,2024010486807,2,2,1,25.0,25.0,6.0,1,1,1,1
1,2024010486821,3,1,0,35.0,20.0,NaN,1,1,1,1
2,2024010486824,2,1,0,40.0,40.0,9.5,1,0,1,0
3,2024010486825,2,2,1,33.0,33.0,NaN,0,0,0,0
4,2024010486828,1,1,0,59.0,59.0,NaN,0,0,0,0


In [13]:
# Aggregate casualties table to collision level
# Casualties table: one row = one casualty
# Need to aggregate to one row = one collision
#
# Important:
# Do NOT use casualty_severity as a predictor for collision_severity.


cas = casualties.copy()

if "age_of_casualty" in cas.columns:
    cas["age_of_casualty_clean"] = pd.to_numeric(cas["age_of_casualty"], errors="coerce")
    cas.loc[cas["age_of_casualty_clean"] < 0, "age_of_casualty_clean"] = np.nan

cas_group = cas.groupby("collision_index")

cas_features = pd.DataFrame({
    "collision_index": cas_group.size().index,
    "casualty_rows": cas_group.size().values
})

# Casualty age features
if "age_of_casualty_clean" in cas.columns:
    cas_features = cas_features.merge(
        cas_group["age_of_casualty_clean"].mean().reset_index(name="mean_casualty_age"),
        on="collision_index",
        how="left"
    )

    cas_features = cas_features.merge(
        cas_group["age_of_casualty_clean"].min().reset_index(name="min_casualty_age"),
        on="collision_index",
        how="left"
    )

# Pedestrian casualty
if "casualty_class" in cas.columns:
    # casualty_class: 1 = driver/rider, 2 = passenger, 3 = pedestrian
    cas["has_pedestrian_casualty"] = (cas["casualty_class"] == 3).astype(int)

    cas_features = cas_features.merge(
        cas_group["has_pedestrian_casualty"].max().reset_index(),
        on="collision_index",
        how="left"
    )

# Casualty type diversity and vulnerable road user
if "casualty_type" in cas.columns:
    cas_features = cas_features.merge(
        cas_group["casualty_type"].nunique().reset_index(name="n_casualty_types"),
        on="collision_index",
        how="left"
    )

    # Common STATS19 casualty_type coding:
    # 0 = pedestrian, 1 = cyclist, 2-5 = motorcyclists
    cas["has_vulnerable_road_user"] = cas["casualty_type"].isin([0, 1, 2, 3, 4, 5]).astype(int)

    cas_features = cas_features.merge(
        cas_group["has_vulnerable_road_user"].max().reset_index(),
        on="collision_index",
        how="left"
    )

print("Casualty features:", cas_features.shape)
cas_features.head()

Casualty features: (100927, 7)


,collision_index,casualty_rows,mean_casualty_age,min_casualty_age,has_pedestrian_casualty,n_casualty_types,has_vulnerable_road_user
0,2024010486807,1,25.0,25.0,0,1,1
1,2024010486821,2,20.0,20.0,0,1,0
2,2024010486824,1,40.0,40.0,0,1,0
3,2024010486825,1,33.0,33.0,0,1,1
4,2024010486828,1,39.0,39.0,1,1,1


In [14]:

# Merge collisions + vehicle features + casualty features

df_master = df_col_clean.merge(
    veh_features,
    on="collision_index",
    how="left"
)

df_master = df_master.merge(
    cas_features,
    on="collision_index",
    how="left"
)

print("Cleaned collisions:", df_col_clean.shape)
print("Vehicle features:", veh_features.shape)
print("Casualty features:", cas_features.shape)
print("Master table:", df_master.shape)

df_master.head()

Cleaned collisions: (100921, 54)
Vehicle features: (100927, 11)
Casualty features: (100927, 7)
Master table: (100921, 70)


,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,local_authority_district,local_authority_ons_district,local_authority_highway,local_authority_highway_current,first_road_class,first_road_number,road_type,speed_limit,junction_detail_historic,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control_historic,pedestrian_crossing_physical_facilities_historic,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,severe,hour,month,weekend,night,peak_hour,at_junction,has_pedestrian_crossing_facility,has_special_condition,has_carriageway_hazard,vehicle_rows,n_vehicle_types,has_vulnerable_vehicle,mean_driver_age,min_driver_age,mean_vehicle_age,has_skidding_or_overturning,has_vehicle_leaving_carriageway,has_hit_object_in_carriageway,has_hit_object_off_carriageway,casualty_rows,mean_casualty_age,min_casualty_age,has_pedestrian_casualty,n_casualty_types,has_vulnerable_road_user
0,202417M119024,2024,17M119024,450057,519938,-1.22722,54.57219,17,3,2,1,2024-12-05,5,16:10,-1,E06000002,E06000002,E06000002,6,0,6,30,-1,0,-99,-1,-1,-1,-1,0,4,2,2,-99,-1,0,1,3,2,E01035190,3,1,0.000000,1.000000,0,16,12,0,0,1,-99,0,-99,0,2,1,0,35.0,35.0,7.0,0,0,0,0,1,27.0,27.0,0,1,0
1,202417S312124,2024,17S312124,445858,516830,-1.29265,54.54466,17,3,1,1,2024-10-22,3,14:56,-1,E06000004,E06000004,E06000004,6,0,6,30,-1,0,-99,-1,-1,-1,-1,0,1,1,1,-99,-1,0,1,3,2,E01012239,3,1,0.000000,1.000000,0,14,10,0,0,0,-99,0,-99,0,1,1,0,NaN,NaN,10.0,0,0,0,0,1,11.0,11.0,1,1,1
2,2024070110901,2024,070110901,364765,390979,-2.53157,53.41443,7,3,2,1,2024-01-24,4,17:50,-1,E06000007,E06000007,E06000007,6,0,6,30,3,13,4,6,0,0,0,0,4,1,1,0,0,0,2,2,2,E01012459,-1,0,0.014743,0.985257,0,17,1,0,0,1,1,0,0,0,2,1,0,45.0,45.0,12.5,0,1,0,0,1,45.0,45.0,0,1,0
3,2024041446676,2024,041446676,365989,419804,-2.51630,53.67359,4,2,1,1,2024-05-22,4,17:45,-1,E06000008,E06000008,E06000008,3,675,6,50,-1,0,-99,-1,-1,-1,-1,0,1,5,5,-99,-1,0,2,1,2,E01012628,6,1,1.000000,0.000000,1,17,5,0,0,1,-99,0,-99,0,1,1,0,66.0,66.0,12.0,1,1,0,1,1,66.0,66.0,0,1,0
4,2024041478641,2024,041478641,366864,430071,-2.50416,53.76592,4,2,2,1,2024-08-11,1,15:36,-1,E06000008,E06000008,E06000008,3,6119,3,50,-1,16,4,6,0,-1,-1,0,1,1,1,-99,-1,0,1,1,2,E01012581,7,1,1.000000,0.000000,1,15,8,1,0,0,1,0,-99,0,2,2,1,30.5,24.0,21.0,1,0,0,0,1,24.0,24.0,0,1,1


In [15]:

#  Check merge quality


check_cols = [
    "collision_index",
    "collision_severity",
    "severe",
    "number_of_vehicles",
    "number_of_casualties",
    "vehicle_rows",
    "casualty_rows"
]

check_cols = [c for c in check_cols if c in df_master.columns]

display(df_master[check_cols].head(10))
display(df_master[check_cols].isnull().sum())

,collision_index,collision_severity,severe,number_of_vehicles,number_of_casualties,vehicle_rows,casualty_rows
0,202417M119024,3,0,2,1,2,1
1,202417S312124,3,0,1,1,1,1
2,2024070110901,3,0,2,1,2,1
3,2024041446676,2,1,1,1,1,1
4,2024041478641,2,1,2,1,2,1
5,2024161425653,3,0,2,2,2,2
6,2024161438773,3,0,1,1,1,1
7,2024122400772,3,0,4,1,4,1
8,2024302400537,3,0,2,2,2,2
9,2024302400805,3,0,2,1,2,1


,0
collision_index,0
collision_severity,0
severe,0
number_of_vehicles,0
number_of_casualties,0
vehicle_rows,0
casualty_rows,0


In [16]:

# Create final modelling table


selected_cols = [
    # ID and target
    "collision_index",
    "collision_year",
    "collision_ref_no",
    "collision_severity",
    "severe",

    # Original time
    "date",
    "time",

    # Temporal variables
    "day_of_week",
    "hour",
    "month",
    "weekend",
    "night",
    "peak_hour",

    # Collision-level variables
    "number_of_vehicles",
    "number_of_casualties",

    # Road-context variables
    "road_type",
    "urban_or_rural_area",
    "speed_limit",
    "first_road_class",
    "junction_detail",
    "junction_control",
    "at_junction",
    "pedestrian_crossing",
    "has_pedestrian_crossing_facility",

    # Environmental variables
    "weather_conditions",
    "road_surface_conditions",
    "light_conditions",
    "carriageway_hazards",
    "has_carriageway_hazard",
    "special_conditions_at_site",
    "has_special_condition",

    # Vehicle-derived variables
    "vehicle_rows",
    "n_vehicle_types",
    "has_vulnerable_vehicle",
    "mean_driver_age",
    "min_driver_age",
    "mean_vehicle_age",
    "has_skidding_or_overturning",
    "has_vehicle_leaving_carriageway",
    "has_hit_object_in_carriageway",
    "has_hit_object_off_carriageway",

    # Casualty-derived variables
    "casualty_rows",
    "mean_casualty_age",
    "min_casualty_age",
    "has_pedestrian_casualty",
    "n_casualty_types",
    "has_vulnerable_road_user"
]

selected_cols = [c for c in selected_cols if c in df_master.columns]

model_df = df_master[selected_cols].copy()

print("Model table:", model_df.shape)
print(model_df.columns.tolist())

model_df.head()

Model table: (100921, 47)
['collision_index', 'collision_year', 'collision_ref_no', 'collision_severity', 'severe', 'date', 'time', 'day_of_week', 'hour', 'month', 'weekend', 'night', 'peak_hour', 'number_of_vehicles', 'number_of_casualties', 'road_type', 'urban_or_rural_area', 'speed_limit', 'first_road_class', 'junction_detail', 'junction_control', 'at_junction', 'pedestrian_crossing', 'has_pedestrian_crossing_facility', 'weather_conditions', 'road_surface_conditions', 'light_conditions', 'carriageway_hazards', 'has_carriageway_hazard', 'special_conditions_at_site', 'has_special_condition', 'vehicle_rows', 'n_vehicle_types', 'has_vulnerable_vehicle', 'mean_driver_age', 'min_driver_age', 'mean_vehicle_age', 'has_skidding_or_overturning', 'has_vehicle_leaving_carriageway', 'has_hit_object_in_carriageway', 'has_hit_object_off_carriageway', 'casualty_rows', 'mean_casualty_age', 'min_casualty_age', 'has_pedestrian_casualty', 'n_casualty_types', 'has_vulnerable_road_user']


,collision_index,collision_year,collision_ref_no,collision_severity,severe,date,time,day_of_week,hour,month,weekend,night,peak_hour,number_of_vehicles,number_of_casualties,road_type,urban_or_rural_area,speed_limit,first_road_class,junction_detail,junction_control,at_junction,pedestrian_crossing,has_pedestrian_crossing_facility,weather_conditions,road_surface_conditions,light_conditions,carriageway_hazards,has_carriageway_hazard,special_conditions_at_site,has_special_condition,vehicle_rows,n_vehicle_types,has_vulnerable_vehicle,mean_driver_age,min_driver_age,mean_vehicle_age,has_skidding_or_overturning,has_vehicle_leaving_carriageway,has_hit_object_in_carriageway,has_hit_object_off_carriageway,casualty_rows,mean_casualty_age,min_casualty_age,has_pedestrian_casualty,n_casualty_types,has_vulnerable_road_user
0,202417M119024,2024,17M119024,3,0,2024-12-05,16:10,5,16,12,0,0,1,2,1,6,1,30,6,0,-99,-99,0,0,2,2,4,0,0,-99,-99,2,1,0,35.0,35.0,7.0,0,0,0,0,1,27.0,27.0,0,1,0
1,202417S312124,2024,17S312124,3,0,2024-10-22,14:56,3,14,10,0,0,0,1,1,6,1,30,6,0,-99,-99,0,0,1,1,1,0,0,-99,-99,1,1,0,NaN,NaN,10.0,0,0,0,0,1,11.0,11.0,1,1,1
2,2024070110901,2024,070110901,3,0,2024-01-24,17:50,4,17,1,0,0,1,2,1,6,2,30,6,13,4,1,0,0,1,1,4,0,0,0,0,2,1,0,45.0,45.0,12.5,0,1,0,0,1,45.0,45.0,0,1,0
3,2024041446676,2024,041446676,2,1,2024-05-22,17:45,4,17,5,0,0,1,1,1,6,2,50,3,0,-99,-99,0,0,5,5,1,0,0,-99,-99,1,1,0,66.0,66.0,12.0,1,1,0,1,1,66.0,66.0,0,1,0
4,2024041478641,2024,041478641,2,1,2024-08-11,15:36,1,15,8,1,0,0,2,1,3,1,50,3,16,4,1,0,0,1,1,1,0,0,-99,-99,2,2,1,30.5,24.0,21.0,1,0,0,0,1,24.0,24.0,0,1,1


Rightthere I found the unpredicted missing value(NaN) in ‘age’ varible.

In [17]:

#  Missing-value summary


missing_summary = model_df.isnull().mean().sort_values(ascending=False).reset_index()
missing_summary.columns = ["column", "missing_ratio"]

missing_summary

,column,missing_ratio
0,mean_vehicle_age,0.158877
1,mean_driver_age,0.056064
2,min_driver_age,0.056064
3,mean_casualty_age,0.015795
4,min_casualty_age,0.015795
5,collision_year,0.000000
6,collision_index,0.000000
7,day_of_week,0.000000
8,hour,0.000000
9,month,0.000000


I quote the ‘missing summary’ right here and quite sure about there are unpredict missing value, so I prepare to exclude them from the final modelling datasets because they introduce additional missing-value handling and are not required for the main comparison.

In [18]:
model_df.to_csv("road_safety_2024_collision_level_model_table.csv", index=False)
df_master.to_csv("road_safety_2024_master_full_table.csv", index=False)
cleaning_report.to_csv("collision_cleaning_report_2024.csv", index=False)
missing_summary.to_csv("missing_summary_2024.csv", index=False)

print("Exported:")
print("road_safety_2024_collision_level_model_table.csv")
print("road_safety_2024_master_full_table.csv")
print("collision_cleaning_report_2024.csv")


Exported:
road_safety_2024_collision_level_model_table.csv
road_safety_2024_master_full_table.csv
collision_cleaning_report_2024.csv


In [19]:
from google.colab import files

files.download("road_safety_2024_collision_level_model_table.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
# Model A: Baseline road-environment model

baseline_cols = [
    "severe",
    "speed_limit",
    "weather_conditions",
    "road_surface_conditions",
    "light_conditions"
]

baseline_cols = [c for c in baseline_cols if c in df_master.columns]
baseline_model_df = df_master[baseline_cols].copy()

print("Baseline road-environment model table:", baseline_model_df.shape)
print(baseline_model_df.columns.tolist())

display(baseline_model_df.head(10))

print("\nMissing values in baseline table:")
display(baseline_model_df.isnull().sum().sort_values(ascending=False))


Baseline road-environment model table: (100921, 5)
['severe', 'speed_limit', 'weather_conditions', 'road_surface_conditions', 'light_conditions']


,severe,speed_limit,weather_conditions,road_surface_conditions,light_conditions
0,0,30,2,2,4
1,0,30,1,1,1
2,0,30,1,1,4
3,1,50,5,5,1
4,1,50,1,1,1
5,0,40,-99,1,1
6,0,30,1,1,1
7,0,30,1,1,1
8,0,30,1,1,1
9,0,40,1,1,1



Missing values in baseline table:


,0
severe,0
speed_limit,0
weather_conditions,0
road_surface_conditions,0
light_conditions,0


In [25]:
# Model B: Compact extended collision-road model


compact_extended_cols = [
    # target
    "severe",

    # collision-level characteristics
    "number_of_vehicles",
    "number_of_casualties",

    # road-context characteristics
    "speed_limit",
    "road_type",
    "urban_or_rural_area",
    "junction_detail",

    # temporal characteristics
    "day_of_week",
    "hour",

    # road-environment and scene-condition variables
    "weather_conditions",
    "road_surface_conditions",
    "light_conditions",
    "has_carriageway_hazard",

    # vehicle / road-user involvement indicators
    "has_vulnerable_vehicle",
    "has_skidding_or_overturning",
    "has_pedestrian_casualty",
    "has_vulnerable_road_user"
]

compact_extended_cols = [c for c in compact_extended_cols if c in df_master.columns]
compact_extended_model_df = df_master[compact_extended_cols].copy()

print("\nCompact extended collision-road model table:", compact_extended_model_df.shape)
print(compact_extended_model_df.columns.tolist())

display(compact_extended_model_df.head(10))

print("\nMissing values in compact extended table:")
display(compact_extended_model_df.isnull().sum().sort_values(ascending=False))


Compact extended collision-road model table: (100921, 17)
['severe', 'number_of_vehicles', 'number_of_casualties', 'speed_limit', 'road_type', 'urban_or_rural_area', 'junction_detail', 'day_of_week', 'hour', 'weather_conditions', 'road_surface_conditions', 'light_conditions', 'has_carriageway_hazard', 'has_vulnerable_vehicle', 'has_skidding_or_overturning', 'has_pedestrian_casualty', 'has_vulnerable_road_user']


,severe,number_of_vehicles,number_of_casualties,speed_limit,road_type,urban_or_rural_area,junction_detail,day_of_week,hour,weather_conditions,road_surface_conditions,light_conditions,has_carriageway_hazard,has_vulnerable_vehicle,has_skidding_or_overturning,has_pedestrian_casualty,has_vulnerable_road_user
0,0,2,1,30,6,1,0,5,16,2,2,4,0,0,0,0,0
1,0,1,1,30,6,1,0,3,14,1,1,1,0,0,0,1,1
2,0,2,1,30,6,2,13,4,17,1,1,4,0,0,0,0,0
3,1,1,1,50,6,2,0,4,17,5,5,1,0,0,1,0,0
4,1,2,1,50,3,1,16,1,15,1,1,1,0,1,1,0,1
5,0,2,2,40,6,2,0,3,14,-99,1,1,0,0,0,0,0
6,0,1,1,30,6,1,0,6,8,1,1,1,0,0,0,1,1
7,0,4,1,30,6,1,0,5,8,1,1,1,0,0,1,0,0
8,0,2,2,30,1,1,0,5,8,1,1,1,0,0,0,0,0
9,0,2,1,40,1,1,0,6,17,1,1,1,0,0,0,0,0



Missing values in compact extended table:


,0
severe,0
number_of_vehicles,0
number_of_casualties,0
speed_limit,0
road_type,0
urban_or_rural_area,0
junction_detail,0
day_of_week,0
hour,0
weather_conditions,0


Double check with the variable, there are no missing value which is perfect， but there are 29 variables which is a littele bit too much, I decide to promote my modle.

Previous crash severity studies often use a broad range of explanatory variables, including crash-level, road-context, temporal, environmental, vehicle, and road-user factors. However, to keep the model interpretable for this project, I use a compact extended feature set rather than including all available variables. The baseline model uses four selected road-environmental predictors, while the extended model uses 14 predictors grouped into collision-level, road-context, temporal, road-environment/scene-condition, and vehicle/road-user involvement variables.

Therfore upon further consideration, since `junction_detail` is already available, `at_junction` may not be necessary. Given the availability of hourly and weekend data, `peak_hour` is not needed for now. With `has_vulnerable_vehicle` and `has_pedestrian_casualty`, a large number of vehicle impact indicators are not required at this stage.

In [26]:
baseline_model_df.to_csv(
    "road_safety_2024_baseline_road_environment_model.csv",
    index=False
)

compact_extended_model_df.to_csv(
    "road_safety_2024_compact_extended_model.csv",
    index=False
)

print("Exported final modelling datasets:")
print("road_safety_2024_baseline_road_environment_model.csv")
print("road_safety_2024_compact_extended_model.csv")

Exported final modelling datasets:
road_safety_2024_baseline_road_environment_model.csv
road_safety_2024_compact_extended_model.csv


In [27]:
from google.colab import files

files.download("road_safety_2024_baseline_road_environment_model.csv")
files.download("road_safety_2024_compact_extended_model.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>